# Sesión 6 — Visualización de datos con IA (Python)

## De los conteos a la imagen

En la sesión anterior contamos palabras y miramos patrones. Eso ya es mucho, pero un conteo en una tabla rara vez convence a nadie. Cuando vos estudiás un conflicto —acá, un conflicto portuario de plantas pesqueras narrado por la prensa— necesitás MOSTRAR lo que encontraste: que el paro tuvo un pico, que ciertas palabras dominan el relato, que el tiempo tiene una forma.

Una aclaración antes de arrancar: el corpus `../datos/notas_prensa.csv` es SINTÉTICO, ficticio, inspirado en la prensa y la conflictividad obrera del Río de la Plata. Sirve para aprender el método sin pisar callos de nadie. El método, eso sí, es el mismo que vas a usar con tus fuentes reales.

Una idea que quiero que te lleves de esta sesión, por encima de cualquier línea de código: **un gráfico es un argumento, no un adorno**. Cada vez que elegís barras, una línea o una nube de palabras, estás eligiendo QUÉ querés que se vea. Esa decisión es tuya, de quien investiga. La IA dibuja; vos decidís qué se muestra y, sobre todo, validás que lo que se muestra sea cierto.

Vamos a hacer tres gráficos sobre el mismo corpus:

1. **Barras** con el top 10 de palabras más frecuentes (comparar magnitudes).
2. **Serie temporal** con la cantidad de notas por semana (ver evolución en el tiempo).
3. **Nube de palabras** para una mirada panorámica del vocabulario.

Cada uno responde una pregunta distinta. Ahí está la clave: primero la pregunta, después el gráfico.

## Preparar el terreno

Para la nube de palabras vamos a usar la librería `wordcloud`, que no viene con Python por defecto. Si no la tenés, instalala una sola vez (desde la terminal, o en una celda con `!pip install wordcloud`):

```
pip install wordcloud
```

> **El prompt:**
> "En Python, cargá `../datos/notas_prensa.csv` con pandas y mostrame las primeras filas para confirmar que están las columnas id, fecha, diario, seccion, titular y texto."

In [ ]:
# pandas para tablas, matplotlib para graficar
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re

# Leemos el corpus a un DataFrame
notas = pd.read_csv("../datos/notas_prensa.csv")

# Antes de hacer NADA, miramos qué tenemos entre manos
notas.head()

**Qué hace y cómo lo validamos:** leímos el CSV a una tabla (`notas`). Antes de graficar, MIRÁ los datos. ¿Aparecen las seis columnas? ¿La fecha tiene pinta de fecha (AAAA-MM-DD)? Si algo no cuadra acá, todo lo que viene después va a estar mal. La validación empieza por mirar la fuente, no por confiar en la salida.

## Gráfico 1 — Barras: el top 10 de palabras

La pregunta humanística: **¿qué vocabulario domina el relato del conflicto?** Si las palabras más frecuentes son "paro", "huelga", "asamblea", la prensa está contando una cosa; si fueran "acuerdo", "diálogo", "negociación", contaría otra. El gráfico de barras sirve cuando querés COMPARAR magnitudes entre categorías: acá, cuántas veces aparece cada palabra.

> **El prompt:**
> "Sobre la columna `texto`, pasá todo a minúsculas, separá en palabras, sacá las palabras vacías en español (stopwords), contá las 10 más frecuentes y hacé un gráfico de barras horizontales ordenadas de mayor a menor con matplotlib."

In [ ]:
# Lista corta de palabras vacías (artículos, preposiciones, conectores).
# La armamos a mano para no depender de librerías extra.
vacias = {
    "el", "la", "los", "las", "un", "una", "unos", "unas", "de", "del", "a", "ante",
    "con", "por", "para", "en", "y", "o", "u", "que", "se", "su", "sus", "al", "lo",
    "le", "les", "es", "son", "fue", "ser", "mas", "más", "no", "sin", "entre", "sobre",
    "este", "esta", "estos", "estas", "como", "ya", "si", "tras", "hay"
}

# Pegamos todos los textos en uno solo y lo pasamos a minúsculas
texto_todo = " ".join(notas["texto"]).lower()

# re.findall separa en palabras (incluye acentos y la eñe)
palabras = re.findall(r"[a-záéíóúñ]+", texto_todo)

# Sacamos las vacías
palabras = [p for p in palabras if p not in vacias]

# Contamos y nos quedamos con las 10 de arriba
top10 = Counter(palabras).most_common(10)
top10

In [ ]:
# Separamos etiquetas y valores. Invertimos para que la mayor quede ARRIBA.
etiquetas = [palabra for palabra, _ in top10][::-1]
valores = [n for _, n in top10][::-1]

plt.figure(figsize=(8, 5))
plt.barh(etiquetas, valores, color="#2c6e91")
plt.title("Las 10 palabras más frecuentes en la cobertura del conflicto")
plt.xlabel("Frecuencia (cantidad de apariciones)")
plt.figtext(0.99, 0.01, "Fuente: corpus de ejemplo del curso",
            ha="right", fontsize=8, color="gray")
plt.tight_layout()
plt.show()

**Qué hace y cómo lo validamos:** separamos el texto en palabras, sacamos las vacías, contamos y graficamos las 10 que más aparecen. Las barras son horizontales y están ORDENADAS a propósito: el ojo compara largos mucho mejor cuando están alineados y rankeados. Para validar, mirá la lista `top10` ANTES del gráfico: ¿el orden de las barras coincide con los números? ¿Aparecen palabras que esperabas para un conflicto portuario (paro, huelga, sindicato, trabajadores)? Si se colara una palabra vacía, la agregás al conjunto `vacias`. El criterio de qué es "ruido" lo ponés vos.

## Gráfico 2 — Serie temporal: notas por semana

La pregunta humanística: **¿cómo se distribuyó la cobertura en el tiempo?** Un conflicto tiene ritmo: tensión, asamblea, estallido del paro, negociación, acuerdo. Si la prensa publicó más notas en ciertas semanas, eso marca los momentos calientes. La serie temporal sirve cuando el dato tiene un eje de tiempo y querés ver su EVOLUCIÓN.

> **El prompt:**
> "Convertí la columna `fecha` a tipo fecha, agrupá las notas por semana y hacé una serie temporal (línea con puntos) que muestre cuántas notas se publicaron en cada semana."

In [ ]:
# Convertimos la columna fecha a tipo datetime de verdad
notas["fecha"] = pd.to_datetime(notas["fecha"])

# Agrupamos por semana (W-MON = semanas que terminan el lunes) y contamos
por_semana = notas.resample("W-MON", on="fecha").size()

por_semana

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(por_semana.index, por_semana.values,
         marker="o", color="#2c6e91", linewidth=2)
plt.title("Cantidad de notas publicadas por semana")
plt.xlabel("Semana")
plt.ylabel("Notas publicadas")
plt.xticks(rotation=45, ha="right")
plt.figtext(0.99, 0.01, "Fuente: corpus de ejemplo del curso",
            ha="right", fontsize=8, color="gray")
plt.tight_layout()
plt.show()

**Qué hace y cómo lo validamos:** convertimos la fecha, agrupamos por semana y dibujamos una línea con puntos. La línea sugiere continuidad en el tiempo; los puntos marcan cada observación real (que no haya magia entre semanas). Validación a mano: sumá los valores de `por_semana` y tiene que dar el total de filas del CSV (22 notas) — probá con `por_semana.sum()`. Si una semana tiene un pico, andá al CSV y fijate qué pasó esos días: vas a encontrar el paro y la asamblea multitudinaria. El gráfico te lleva a la fuente; nunca la reemplaza.

## Gráfico 3 — Nube de palabras

La pregunta humanística: **¿qué impresión general da el vocabulario del corpus de un solo vistazo?** La nube de palabras NO es para medir con precisión —para eso ya tenemos las barras—, es para una mirada panorámica, exploratoria. Es seductora y por eso mismo hay que usarla con cuidado: el tamaño impresiona, pero engaña si uno quiere comparar dos palabras parecidas. Sirve para abrir preguntas, no para cerrarlas.

> **El prompt:**
> "Con la librería wordcloud, hacé una nube de palabras a partir de las frecuencias del corpus (sacando las palabras vacías), mostrando las 50 palabras más frecuentes con el tamaño según su frecuencia."

In [ ]:
# wordcloud NO viene con Python. Instalala una vez: pip install wordcloud
from wordcloud import WordCloud

# Reusamos el MISMO conteo de palabras que en las barras (mismo criterio)
frecuencias = dict(Counter(palabras).most_common(50))

# generate_from_frequencies dibuja según el conteo que le pasamos
nube = WordCloud(
    width=800, height=500,
    background_color="white",
    colormap="Blues",
    random_state=42  # fija el azar para que la nube sea reproducible
).generate_from_frequencies(frecuencias)

plt.figure(figsize=(9, 5.5))
plt.imshow(nube, interpolation="bilinear")
plt.axis("off")  # sacamos los ejes: en una nube no significan nada
plt.title("Nube de palabras del corpus")
plt.tight_layout()
plt.show()

**Qué hace y cómo lo validamos:** reusamos el conteo de palabras (mismo criterio que en las barras) y lo mostramos como nube, con el tamaño proporcional a la frecuencia. Fijate que `random_state=42` hace que el acomodo sea siempre igual: sin eso, cada corrida te daría una nube distinta y no podrías reproducir tu figura. Validación: la palabra MÁS grande de la nube tiene que ser la primera de la lista `top10` del gráfico de barras. Si no coinciden, algo se rompió. La nube y las barras cuentan la misma historia desde dos ángulos.

## Para jugar

Probá pedirle a la IA estas variaciones y mirá qué cambia:

- "Hacé el mismo gráfico de barras pero sobre los TITULARES en vez del texto completo. ¿Cambia el ranking? ¿Por qué un titular pesa distinto que el cuerpo de la nota?"
- "En la serie temporal, dibujá una línea por cada diario (`Diario del Puerto`, `La Voz Obrera`, `El Litoral del Sur`) para comparar quién cubrió más y cuándo." Después preguntate: ¿qué argumento sostiene ese gráfico que el anterior no podía?